# Evaluación con Métricas: Decisión Final del Modelo
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

Responder con evidencia cuantitativa (no solo overlap entre modelos) cuál motor de búsqueda usar en el sistema final, comparando:

1. **`all-MiniLM-L6-v2`** (embeddings, elegido preliminarmente)
2. **`all-mpnet-base-v2`** (embeddings, alternativa de mayor calidad/costo)
3. **Baseline léxico (TF-IDF + similitud coseno)** — el método tradicional de coincidencia de palabras clave, tal como pide el Objetivo específico 5 del anteproyecto ("comparando sus resultados con un método tradicional basado en coincidencias léxicas")

usando las métricas estándar de recuperación de información: **Precision@k**, **MRR** y **nDCG@k**, además de tiempo de respuesta.

### Metodología

Como no existen etiquetas de relevancia "verdaderas" para este dataset, se sigue el método estándar en evaluación de sistemas de búsqueda (*pooling*): se juntan los resultados top-k de los tres sistemas para cada consulta de prueba, se eliminan duplicados, y **una persona del equipo etiqueta manualmente** qué tan relevante es cada resultado (sin saber de qué sistema salió, para no sesgar el juicio). Con esas etiquetas ya se pueden calcular las métricas de forma objetiva para los tres sistemas.

Este notebook asume que ya ejecutaste `02_Preprocesamiento.ipynb` y `03_Embeddings_BusquedaVectorial.ipynb` al menos una vez.

## 1. Preparación e imports

In [2]:
!pip install -q sentence-transformers faiss-cpu scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.9 MB/s eta 0:00:00


In [3]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)


## 2. Cargar plantillas y embeddings desde Drive

Se reutiliza el embedding de MiniLM ya calculado y guardado en el notebook 03. El de MPNet se vuelve a generar aquí mismo (son solo 3,760 textos, toma segundos) ya que no se había guardado a disco.

In [4]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
RUTA_PROCESSED = BASE_DIR / "data" / "processed"

plantillas = pd.read_parquet(RUTA_PROCESSED / "plantillas_meta.parquet")
embeddings_minilm = np.load(RUTA_PROCESSED / "embeddings_plantillas.npy")

print(f"Plantillas: {len(plantillas):,}  |  Embeddings MiniLM: {embeddings_minilm.shape}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Plantillas: 3,760  |  Embeddings MiniLM: (3760, 384)


In [5]:
from sentence_transformers import SentenceTransformer

modelo_minilm = SentenceTransformer("all-MiniLM-L6-v2")
modelo_mpnet = SentenceTransformer("all-mpnet-base-v2")

inicio = time.time()
embeddings_mpnet = modelo_mpnet.encode(
    plantillas["texto_combinado"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embeddings MPNet generados en {time.time() - inicio:.1f} s: {embeddings_mpnet.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Embeddings MPNet generados en 30.3 s: (3760, 768)


## 3. Índices FAISS para los dos modelos de embeddings

In [6]:
import faiss

indice_minilm = faiss.IndexFlatIP(embeddings_minilm.shape[1])
indice_minilm.add(embeddings_minilm)

indice_mpnet = faiss.IndexFlatIP(embeddings_mpnet.shape[1])
indice_mpnet.add(embeddings_mpnet)

print(f"Plantillas indexadas -> MiniLM: {indice_minilm.ntotal:,} | MPNet: {indice_mpnet.ntotal:,}")


Plantillas indexadas -> MiniLM: 3,760 | MPNet: 3,760


## 4. Baseline léxico (TF-IDF + similitud coseno)

Representa cada plantilla como un vector de frecuencias de palabras ponderadas (TF-IDF), en vez de un embedding semántico. Es el método "tradicional" contra el que pide comparar el Objetivo específico 5 — no entiende sinónimos ni contexto, solo coincidencia de palabras.

In [7]:
vectorizador_tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
matriz_tfidf = vectorizador_tfidf.fit_transform(plantillas["texto_combinado"])

print(f"Matriz TF-IDF: {matriz_tfidf.shape}")


def buscar_lexico(consulta: str, k: int = 10):
    vector_consulta = vectorizador_tfidf.transform([consulta])
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf).flatten()
    top_k_idx = similitudes.argsort()[::-1][:k]
    return list(zip(plantillas.iloc[top_k_idx]["template_id"].values, similitudes[top_k_idx]))


Matriz TF-IDF: (3760, 2298)


## 5. Consultas de prueba

Se amplía el set inicial de 5 consultas a 15, cubriendo perfiles técnicos, no técnicos, de distintos niveles de seniority, y frases tanto cortas como descriptivas — buscando que la evaluación no dependa de un solo tipo de consulta.

In [8]:
CONSULTAS_EVALUACION = [
    "python developer with machine learning and NLP experience",
    "senior backend engineer with cloud and microservices experience",
    "marketing manager with social media and branding skills",
    "entry level data entry no experience required",
    "financial analyst with excel and forecasting skills",
    "frontend developer react and javascript",
    "human resources recruiter talent acquisition",
    "graphic designer with adobe photoshop and illustrator",
    "sales representative with negotiation skills",
    "network administrator with cisco and security experience",
    "project manager agile scrum certified",
    "customer support representative bilingual",
    "civil engineer with autocad experience",
    "nurse with patient care experience",
    "warehouse worker forklift operation",
]

print(f"Consultas de evaluación: {len(CONSULTAS_EVALUACION)}")


Consultas de evaluación: 15


## 6. Armar el "pool" de candidatos para etiquetar

Para cada consulta, se obtiene el top-10 de cada uno de los tres sistemas (MiniLM, MPNet, léxico), se unen sin duplicados, y se guarda en una tabla — sin indicar de qué sistema vino cada resultado, para que el etiquetado manual sea a ciegas (evita el sesgo de favorecer un sistema porque "ya sabes cuál es").

In [9]:
K_EVALUACION = 10

filas_pool = []

for consulta in CONSULTAS_EVALUACION:
    vector_minilm = modelo_minilm.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices_minilm = indice_minilm.search(vector_minilm, K_EVALUACION)
    templates_minilm = plantillas.iloc[indices_minilm[0]]["template_id"].tolist()

    vector_mpnet = modelo_mpnet.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices_mpnet = indice_mpnet.search(vector_mpnet, K_EVALUACION)
    templates_mpnet = plantillas.iloc[indices_mpnet[0]]["template_id"].tolist()

    resultado_lexico = buscar_lexico(consulta, k=K_EVALUACION)
    templates_lexico = [tid for tid, _ in resultado_lexico]

    plantillas_unicas = sorted(set(templates_minilm + templates_mpnet + templates_lexico))

    for template_id in plantillas_unicas:
        fila_plantilla = plantillas[plantillas["template_id"] == template_id].iloc[0]
        filas_pool.append({
            "consulta": consulta,
            "template_id": template_id,
            "Job Title": fila_plantilla["Job Title"],
            "Role": fila_plantilla["Role"],
            "texto_combinado_snippet": fila_plantilla["texto_combinado"][:200],
            "relevancia": "",  # 0 = no relevante, 1 = parcialmente relevante, 2 = muy relevante
        })

pool_para_etiquetar = pd.DataFrame(filas_pool)
print(f"Total de pares consulta-plantilla a etiquetar: {len(pool_para_etiquetar):,}")
pool_para_etiquetar.head()


Total de pares consulta-plantilla a etiquetar: 309


,consulta,template_id,Job Title,Role,texto_combinado_snippet,relevancia
0,python developer with machine learning and NLP experience,306,Data Scientist,Machine Learning Engineer,Data Scientist Data Scientist Machine Learning Engineer Machine learning algorithms Python programming Data preprocessing Deep learning Model eval...,
1,python developer with machine learning and NLP experience,979,Data Scientist,Machine Learning Engineer,Data Scientist Data Scientist Machine Learning Engineer Machine learning algorithms Python programming Data preprocessing Deep learning Model eval...,
2,python developer with machine learning and NLP experience,1613,Data Scientist,Machine Learning Engineer,Data Scientist Data Scientist Machine Learning Engineer Machine learning algorithms Python programming Data preprocessing Deep learning Model eval...,
3,python developer with machine learning and NLP experience,1783,Data Scientist,Machine Learning Engineer,Data Scientist Data Scientist Machine Learning Engineer Machine learning algorithms Python programming Data preprocessing Deep learning Model eval...,
4,python developer with machine learning and NLP experience,1843,Data Scientist,Machine Learning Engineer,Data Scientist Data Scientist Machine Learning Engineer Machine learning algorithms Python programming Data preprocessing Deep learning Model eval...,


## 7. Exportar el pool para etiquetado manual

Se guarda como CSV en Drive. **Aquí el equipo debe abrir el archivo (en Google Sheets o Excel) y llenar la columna `relevancia` para cada fila**, usando esta escala:

- **0** — no relacionado con la consulta.
- **1** — parcialmente relacionado (mismo área general, pero no es lo que se buscó específicamente).
- **2** — muy relevante (justo lo que un usuario esperaría encontrar).

Recomendación: que **una sola persona** etiquete todo el archivo, para mantener consistencia de criterio entre consultas. Al terminar, guardar el archivo con el mismo nombre en la misma carpeta de Drive (sobrescribiendo), y luego continuar en la sección 8.

In [10]:
RUTA_POOL = RUTA_PROCESSED / "pool_para_etiquetar.csv"
pool_para_etiquetar.to_csv(RUTA_POOL, index=False)
print(f"Pool guardado en: {RUTA_POOL}")
print("Ábrelo, llena la columna 'relevancia' (0/1/2) para cada fila, guarda, y continúa en la sección 8.")


Pool guardado en: /content/drive/MyDrive/proyecto_integrador/data/processed/pool_para_etiquetar.csv
Ábrelo, llena la columna 'relevancia' (0/1/2) para cada fila, guarda, y continúa en la sección 8.


---
## ⏸️ Pausa manual: etiquetado de relevancia

*(Continuar solo después de haber llenado y guardado la columna `relevancia` en el CSV)*

---

## 8. Cargar el pool ya etiquetado

In [11]:
pool_etiquetado = pd.read_csv(RUTA_POOL)

sin_etiquetar = pool_etiquetado["relevancia"].isna().sum()
if sin_etiquetar > 0:
    raise ValueError(
        f"Hay {sin_etiquetar} filas sin etiquetar todavía en {RUTA_POOL}. "
        "Completa la columna 'relevancia' antes de continuar."
    )

pool_etiquetado["relevancia"] = pool_etiquetado["relevancia"].astype(int)
mapa_relevancia = {
    (fila["consulta"], fila["template_id"]): fila["relevancia"]
    for _, fila in pool_etiquetado.iterrows()
}

print(f"Pool etiquetado cargado: {len(pool_etiquetado):,} pares consulta-plantilla.")
print(pool_etiquetado["relevancia"].value_counts().sort_index())


Pool etiquetado cargado: 309 pares consulta-plantilla.
relevancia
0     10
1     89
2    210
Name: count, dtype: int64


## 9. Funciones de métricas

- **Precision@k**: de los k resultados devueltos, qué fracción tiene relevancia > 0.
- **MRR (Mean Reciprocal Rank)**: 1 / posición del primer resultado relevante (0 si no hay ninguno). Premia que lo relevante aparezca temprano.
- **nDCG@k (normalized Discounted Cumulative Gain)**: como Precision, pero pondera más los resultados relevantes que aparecen en las primeras posiciones, y usa el grado de relevancia (0/1/2), no solo relevante/no-relevante.

In [12]:
def obtener_relevancia(consulta: str, template_id: int) -> int:
    return mapa_relevancia.get((consulta, template_id), 0)  # si no fue parte del pool, se asume 0


def precision_en_k(consulta: str, template_ids_ordenados: list, k: int) -> float:
    top_k = template_ids_ordenados[:k]
    relevantes = sum(1 for tid in top_k if obtener_relevancia(consulta, tid) > 0)
    return relevantes / k


def mrr(consulta: str, template_ids_ordenados: list) -> float:
    for posicion, tid in enumerate(template_ids_ordenados, start=1):
        if obtener_relevancia(consulta, tid) > 0:
            return 1 / posicion
    return 0.0


def ndcg_en_k(consulta: str, template_ids_ordenados: list, k: int) -> float:
    top_k = template_ids_ordenados[:k]
    dcg = sum(
        obtener_relevancia(consulta, tid) / np.log2(posicion + 1)
        for posicion, tid in enumerate(top_k, start=1)
    )
    relevancias_ideales = sorted(
        [obtener_relevancia(consulta, tid) for tid in top_k], reverse=True
    )
    idcg = sum(
        rel / np.log2(posicion + 1)
        for posicion, rel in enumerate(relevancias_ideales, start=1)
    )
    return dcg / idcg if idcg > 0 else 0.0


## 10. Calcular métricas para los tres sistemas

Para cada consulta y cada sistema, se recupera el top-10 (mismo `K_EVALUACION` de la sección 6, para que sea comparable con lo etiquetado) y se calculan las tres métricas, además del tiempo de la consulta.

In [13]:
def evaluar_sistema(nombre_sistema: str, funcion_busqueda):
    filas_metricas = []
    for consulta in CONSULTAS_EVALUACION:
        inicio = time.time()
        template_ids_ordenados = funcion_busqueda(consulta)
        tiempo = time.time() - inicio

        filas_metricas.append({
            "sistema": nombre_sistema,
            "consulta": consulta,
            "precision@5": precision_en_k(consulta, template_ids_ordenados, 5),
            "precision@10": precision_en_k(consulta, template_ids_ordenados, K_EVALUACION),
            "mrr": mrr(consulta, template_ids_ordenados),
            "ndcg@10": ndcg_en_k(consulta, template_ids_ordenados, K_EVALUACION),
            "tiempo_ms": tiempo * 1000,
        })
    return pd.DataFrame(filas_metricas)


def busqueda_minilm(consulta):
    vector = modelo_minilm.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices = indice_minilm.search(vector, K_EVALUACION)
    return plantillas.iloc[indices[0]]["template_id"].tolist()


def busqueda_mpnet(consulta):
    vector = modelo_mpnet.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices = indice_mpnet.search(vector, K_EVALUACION)
    return plantillas.iloc[indices[0]]["template_id"].tolist()


def busqueda_lexica(consulta):
    return [tid for tid, _ in buscar_lexico(consulta, k=K_EVALUACION)]


metricas_minilm = evaluar_sistema("MiniLM", busqueda_minilm)
metricas_mpnet = evaluar_sistema("MPNet", busqueda_mpnet)
metricas_lexico = evaluar_sistema("Léxico (TF-IDF)", busqueda_lexica)

metricas_completas = pd.concat([metricas_minilm, metricas_mpnet, metricas_lexico], ignore_index=True)
metricas_completas.head()


,sistema,consulta,precision@5,precision@10,mrr,ndcg@10,tiempo_ms
0,MiniLM,python developer with machine learning and NLP experience,1.0,1.0,1.0,1.000000,12.686253
1,MiniLM,senior backend engineer with cloud and microservices experience,1.0,1.0,1.0,1.000000,8.211374
2,MiniLM,marketing manager with social media and branding skills,1.0,1.0,1.0,0.919194,7.714033
3,MiniLM,entry level data entry no experience required,1.0,1.0,1.0,1.000000,8.247614
4,MiniLM,financial analyst with excel and forecasting skills,1.0,1.0,1.0,0.993091,9.008884


## 11. Tabla comparativa final

Promedio de cada métrica por sistema, sobre las 15 consultas de evaluación.

In [14]:
resumen_final = (
    metricas_completas
    .groupby("sistema")[["precision@5", "precision@10", "mrr", "ndcg@10", "tiempo_ms"]]
    .mean()
    .round(3)
    .sort_values("ndcg@10", ascending=False)
)

resumen_final


,precision@5,precision@10,mrr,ndcg@10,tiempo_ms
sistema,,,,,
MiniLM,1.000,1.000,1.000,0.986,9.165
MPNet,1.000,1.000,1.000,0.978,16.729
Léxico (TF-IDF),0.933,0.933,0.933,0.933,4.006


In [15]:
# Detalle por consulta, útil para revisar casos puntuales donde un sistema falla
metricas_completas.pivot_table(index="consulta", columns="sistema", values="ndcg@10").round(3)


sistema,Léxico (TF-IDF),MPNet,MiniLM
consulta,,,
civil engineer with autocad experience,1.0,0.877,1.000
customer support representative bilingual,1.0,0.884,1.000
entry level data entry no experience required,1.0,1.000,1.000
financial analyst with excel and forecasting skills,1.0,1.000,0.993
frontend developer react and javascript,1.0,1.000,1.000
graphic designer with adobe photoshop and illustrator,1.0,1.000,0.993
human resources recruiter talent acquisition,1.0,1.000,1.000
marketing manager with social media and branding skills,1.0,1.000,0.919
network administrator with cisco and security experience,1.0,1.000,1.000


## 12. Decisión final

Modelo elegido: all-MiniLM-L6-v2. Iguala a MPNet en precisión y MRR, lo supera ligeramente en nDCG@10, y es sustancialmente más rápido — no hay trade-off real que justifique el modelo más pesado en esta iteración.